# PiShield — encouraging satisfaction with a Shield Loss

This notebook runs **end-to-end with no external downloads**. The *Shield Loss* is an
alternative to the Shield Layer for **propositional** requirements. Instead of *correcting*
the outputs, it adds a differentiable penalty (via a t-norm) that *encourages* the network
to satisfy the requirements. It does not *guarantee* satisfaction, but it can be added
directly to any task loss.

Our requirement: **if variable 0 is true, then variable 1 must be true** (`0 -> 1`).

In [ ]:
import torch
from pishield.shield_loss import build_shield_loss

torch.manual_seed(0)

## 1. Write the requirements file (Horn-rule format)

The Shield Loss reads propositional requirements as `<id> <head> :- <body>`, where literals
are variable indices and an `n` prefix denotes negation. The implication `0 -> 1` is the
Horn rule with head `1` and body `0`.

In [ ]:
requirements_path = "implication_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("c0 1 :- 0\n")   # variable 0 implies variable 1

print(open(requirements_path).read())

## 2. Build the Shield Loss

`tnorm_choice` selects the fuzzy-logic semantics: `'product'`, `'godel'` or `'lukasiewicz'`.

In [ ]:
num_variables = 2
shield_loss = build_shield_loss(num_variables, requirements_path, tnorm_choice="product")

## 3. Minimise the loss

We start from logits that **violate** the rule — high probability for variable 0, low for
variable 1 — and minimise the Shield Loss. Note that the loss takes *probabilities* in
`[0, 1]`, so we pass `sigmoid(logits)`.

In [ ]:
logits = torch.tensor([[3.0, -3.0]], requires_grad=True)  # P(0)~0.95, P(1)~0.05  ->  violates 0 -> 1
opt = torch.optim.SGD([logits], lr=1.0)

for step in range(300):
    probs = torch.sigmoid(logits)
    loss = shield_loss(probs)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 60 == 0:
        p = [round(v, 2) for v in probs.detach().squeeze().tolist()]
        print(f"step {step:3d}  loss={loss.item():.4f}  probs={p}")

print("\nfinal probs:", [round(v, 2) for v in torch.sigmoid(logits).detach().squeeze().tolist()])

## 4. Takeaway

The loss decreases and the probabilities move to satisfy `0 -> 1` (either `P(0)` drops or
`P(1)` rises). In a real model you would add it to your task loss, e.g.
`total_loss = task_loss + shield_loss(probs)`.

Use the **Shield Loss** when you want a soft nudge towards the requirements, and the
**Shield Layer** (see `shield_layer_inference.ipynb` / `shield_layer_training.ipynb`) when you need a hard guarantee.